# 3. Investigate misses, early warnings and operational workload

Do not optimise only an aggregate recall number. Review fault type, magnitude,
missingness, repeated warnings and whether an observable precursor exists.
Sudden loss cannot generally be predicted from these two signals.
Synthetic impact is a simulation proxy, not a real customer SLA.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from telco_anomaly.synthetic import load_config
from telco_anomaly.experiment import split_times, load_data

CONFIG, relative_path = load_config(ROOT / "configs/synthetic.yml")
DATA = ROOT / relative_path
RUN = ROOT / "outputs/simple_model"


In [ ]:
faults = pd.read_csv(RUN / "ewma_faults.csv")
display(pd.read_csv(RUN / "ewma_by_type.csv"))
display(faults.loc[~faults.detected])
display(faults.loc[faults.has_impact,
                   ["fault_id", "type", "detected", "early", "lead_hours"]])
alerts = pd.read_csv(RUN / "ewma_warnings.csv")
display(alerts.head(15))
if faults.detected.any():
    faults.loc[faults.detected, "delay_from_onset_hours"].hist(bins=15)
    plt.xlabel("Hours from physical onset to warning, detected events only")
    plt.show()


## Stress tests use unchanged detector settings

A new seed, noisier/more incomplete observations, hourly sampling and a fault-free
scenario challenge different assumptions. They remain variations of the same
simulator, not external validation. Running this cell with the flag enabled generates
four additional datasets and may take a few minutes. It refuses to overwrite results.

In [ ]:
from telco_anomaly.experiment import robustness_suite, final_evaluation

RUN_STRESS_TESTS = False
STRESS_RUN = ROOT / "outputs/simple_stress"
if RUN_STRESS_TESTS:
    display(robustness_suite(CONFIG, STRESS_RUN))


## Final assessment is an explicit decision

Set the flag only after fixing the model and reviewing development errors.
The helper checks the saved code, data and model fingerprints, then records that
final assessment was opened. It refuses a second opening in this run folder.
This prevents accidental reuse; it is not a security boundary. Once inspected,
the period is no longer an untouched test for further tuning.

In [ ]:
OPEN_FINAL_TEST = False
if OPEN_FINAL_TEST:
    display(pd.Series(final_evaluation(RUN)))
else:
    print("Final performance assessment remains unopened.")
